<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/05-data-sources-io/01-schema-evolution-and-merging.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 5.1 — Schema evolution and merging

We simulate a dataset written over three months by evolving code, then
read the mess every way the lesson describes.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("5.1-schema-evolution")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Build the drifting dataset: three months, three schemas

In [ ]:
# January: (id, amount)
jan = spark.createDataFrame([(1, 10.0), (2, 20.0)], "id INT, amount DOUBLE")
jan.write.mode("overwrite").parquet("output/sales/month=jan")

# February: added 'currency' (additive — the good kind)
feb = spark.createDataFrame([(3, 30.0, "EUR"), (4, 40.0, "USD")],
                            "id INT, amount DOUBLE, currency STRING")
feb.write.mode("overwrite").parquet("output/sales/month=feb")

# March: renamed user column story — wrote 'cust' where jan/feb had none,
# to give the rename/coalesce demo something to chew on
mar = spark.createDataFrame([(5, 50.0, "EUR", "c001")],
                            "id INT, amount DOUBLE, currency STRING, cust STRING")
mar.write.mode("overwrite").parquet("output/sales/month=mar")
print("three schema versions written")

## Plain read: schema comes from a SAMPLE of files — roll the dice

In [ ]:
plain = spark.read.parquet("output/sales")
plain.printSchema()   # does it have currency? cust? depends which file(s) it sampled
plain.show()

## mergeSchema: union of all file schemas (pay the footer scan)

In [ ]:
merged = spark.read.option("mergeSchema", True).parquet("output/sales")
merged.printSchema()
merged.orderBy("id").show()   # missing columns = null, by NAME

## The production stance: one explicit contract schema

In [ ]:
CONTRACT = "id INT, amount DOUBLE, currency STRING, cust STRING"
contracted = spark.read.schema(CONTRACT).parquet("output/sales")
contracted.orderBy("id").show()   # same result as merge, deterministic, no footer scan

# drift detector (lessons 3.5 + 4.2): a contract column that's 100% null is a red flag
total = contracted.count()
contracted.select([
    F.round(F.count(F.when(col(c).isNull(), 1)) / total, 2).alias(c)
    for c in contracted.columns
]).show()

## The rename problem, and the coalesce fix

In [ ]:
# April: 'cust' renamed to 'customer_id' — by-name matching sees TWO columns now
apr = spark.createDataFrame([(6, 60.0, "GBP", "c002")],
                            "id INT, amount DOUBLE, currency STRING, customer_id STRING")
apr.write.mode("overwrite").parquet("output/sales/month=apr")

hist = spark.read.schema(CONTRACT + ", customer_id STRING").parquet("output/sales")
hist.select("id", "cust", "customer_id").orderBy("id").show()   # split brain

fixed = hist.withColumn("customer", F.coalesce(col("customer_id"), col("cust"))) \
            .drop("cust", "customer_id")
fixed.orderBy("id").show()

## CSV: positional matching — the silent column shift

In [ ]:
import pathlib
pathlib.Path("output/csvdrift").mkdir(parents=True, exist_ok=True)
pathlib.Path("output/csvdrift/old.csv").write_text("1,99.0,IN\n", encoding="utf-8")
# producer inserted 'discount' as column 2:
pathlib.Path("output/csvdrift/new.csv").write_text("2,0.15,88.0,DE\n", encoding="utf-8")

spark.read.schema("id INT, amount DOUBLE, country STRING") \
    .csv("output/csvdrift").show()
# row 2: amount=0.15 (the discount!), country='88.0' — wrong values, zero errors

## Cleanup

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)
print("cleaned up.")

## Exercises

1. Rebuild the sales dirs, then write a May file where `amount` is a STRING ("55.0"). What do plain read, mergeSchema, and the contract read each do? (Type conflicts are pain-rank #2 for a reason.)
2. Wrap the null-ratio drift detector into `assert_no_dead_columns(df, threshold=0.99)` and make it raise on the split-brain `hist` DataFrame.
3. JSON edition: read `data/events.jsonl` (DROPMALFORMED) with an explicit schema containing a field NO event has (`session_id STRING`). Confirm the by-name null behavior matches Parquet's.
4. Time the difference: generate 200 tiny parquet files in one dir, compare wall-clock of a plain `.parquet()` load vs `mergeSchema=True` (just `df.schema`, no action needed — why does even THAT cost?).

In [ ]:
# your exercise code here
